In [7]:
!pip install yfinance
import numpy as np
import pandas as pd
import yfinance as yf #yahoo finance

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 14.6 MB/s  0:00:00

   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   -------- ------------------------------- 1/5 [websockets]
   ---------------- ----------------------- 2/5 [peewee]
   ---

In [8]:
actifs=["AAPL", "MSFT", "AMZN", "^GSPC", "TTE.PA"] #TTE.PA est TotalEnergies qui est cotée sur Euronext Paris (.PA)
start_date="2005-01-01"
end_date="2026-01-01"

In [20]:
# Télécharger les prix de clôture ajustés
data=yf.download(actifs,start_date,end_date)["Close"]
data.head()

[*********************100%***********************]  5 of 5 completed


Ticker,AAPL,AMZN,MSFT,TTE.PA,^GSPC
Date,,,,,
2005-01-03,0.946490,2.2260,18.304285,11.815637,1202.079956
2005-01-04,0.956211,2.1070,18.372736,11.771494,1188.050049
2005-01-05,0.964585,2.0885,18.331669,11.661132,1183.739990
2005-01-06,0.965334,2.0525,18.311123,11.749417,1187.890015
2005-01-07,1.035621,2.1160,18.256367,11.786204,1186.189941


In [24]:
# 2. Rendements journaliers
daily_returns = data.pct_change(fill_method=None).dropna() 

In [25]:
# 3. Rendements cumulés (performance nette)
cumulative_returns = (1 + daily_returns).cumprod() - 1

In [26]:
# 4. Volatilité annualisée (252 jours de bourse par an)
annualized_volatility = daily_returns.std() * np.sqrt(252)

In [27]:
# 5. Maximum Drawdown (MDD)
rolling_max = (1 + daily_returns).cumprod().cummax()
drawdown = (1 + daily_returns).cumprod() / rolling_max - 1
max_drawdown = drawdown.min()

In [29]:
annualized_return=daily_returns.mean()*252
sharpe_ratio = annualized_return / annualized_volatility

In [30]:
# 7. Matrice de corrélation
correlation_matrix = daily_returns.corr()

In [39]:
summary_df = pd.DataFrame(
    {
        "Rendement Annualisé": annualized_return,
        "Volatilité Annualisée": annualized_volatility,
        "Ratio de Sharpe": sharpe_ratio,
        "Max Drawdown": max_drawdown,
    }
)
print("--- Résumé des Performances ---")
print(summary_df)
print("\n--- Matrice de Corrélation ---")
print(correlation_matrix)

--- Résumé des Performances ---
        Rendement Annualisé  Volatilité Annualisée  Ratio de Sharpe  \
Ticker                                                                
AAPL               0.298318               0.321497         0.927903   
AMZN               0.277627               0.378938         0.732646   
MSFT               0.193594               0.270970         0.714447   
TTE.PA             0.123311               0.264299         0.466557   
^GSPC              0.100246               0.192086         0.521882   

        Max Drawdown  
Ticker                
AAPL       -0.627286  
AMZN       -0.679228  
MSFT       -0.505953  
TTE.PA     -0.558991  
^GSPC      -0.539186  

--- Matrice de Corrélation ---
Ticker      AAPL      AMZN      MSFT    TTE.PA     ^GSPC
Ticker                                                  
AAPL    1.000000  0.461152  0.528583  0.196254  0.644306
AMZN    0.461152  1.000000  0.526992  0.173948  0.582342
MSFT    0.528583  0.526992  1.000000  0.261301  0